# The resistive dissipation wavenumber over time

$k_{\mathrm{diss}} = \langle J^2 \rangle^{1/4} / \eta^{1/2}$ at every output time of each run of the beta scan,
computed in parallel over the times.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pta import config
from pta.fields import current_density, resistive_dissipation_wavenumber
from pta.plotting import save_figure, use_style

use_style()
RESULTS = config.results_dir("jpar_jperp", "Kdiss")

In [ ]:
BETAS_P = [0.0625, 1, 16]             # proton beta of the runs (the beta_scan layout in pta.toml)
RESISTIVITY = 2.5e-4                  # eta of these runs

In [ ]:
def k_diss_series(run):
    """k_diss at every output time of a run, computed in parallel over the times."""
    def k_diss_at(time):
        Jx, Jy, Jz = current_density(*run.load_many(["Bx", "By", "Bz"], time), run.dx, run.dy)
        return resistive_dissipation_wavenumber(np.mean(Jx**2 + Jy**2 + Jz**2), RESISTIVITY)
    return run.times, np.array(run.map(k_diss_at))


k_diss = {beta_p: k_diss_series(config.run("beta_scan", beta_p=beta_p)) for beta_p in BETAS_P}

In [ ]:
for beta_p, (times, values) in k_diss.items():
    fig, ax = plt.subplots()
    ax.plot(times, values)
    ax.set_xlabel("Time")
    ax.set_ylabel(r"$k_{\mathrm{diss}}$")
    ax.set_title(rf"$\beta_p = {beta_p}$")
    save_figure(fig, RESULTS, f"k_diss_vs_time_beta_p{beta_p}")
    plt.show()